# PCA - Projet Final

## Objectif
Ce notebook implemente la partie PCA du projet collaboratif.

## Etat actuel
- Initialisation de l'environnement
- Chargement des donnees
- Inspection rapide avant preprocessing


In [1]:
# Imports essentiels
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')


In [2]:
# Parametres de base
RANDOM_STATE = 42

PROJECT_ROOT = Path('..')
DATA_PATH = PROJECT_ROOT / 'data' / 'city_lifestyle_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data path: {DATA_PATH}')
print(f'Output dir: {OUTPUT_DIR}')


Data path: ..\data\city_lifestyle_dataset.csv
Output dir: ..\outputs


## 1. Chargement et inspection rapide des donnees

Verification rapide avant la reduction de dimension:
- dimensions du dataset
- type des colonnes
- valeurs manquantes


In [3]:
# Chargement du dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset introuvable: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()


Shape: (300, 10)


,city_name,country,population_density,avg_income,internet_penetration,avg_rent,air_quality_index,public_transport_score,happiness_score,green_space_ratio
0,Old Vista,Europe,2775,3850,86.4,1310,43,52.0,8.5,23.8
1,Beachport,Europe,3861,3700,78.1,1330,42,62.8,8.1,33.1
2,Valleyborough,Europe,2562,4310,80.1,1330,39,73.2,8.5,40.2
3,City,Europe,3192,3970,81.2,1480,60,49.2,8.5,43.6
4,Falls,Europe,3496,4320,100.0,1510,64,93.7,8.5,42.5


In [4]:
# Inspection rapide
print('\nTypes de colonnes:')
print(df.dtypes)

missing = df.isna().sum()
print('\nValeurs manquantes par colonne:')
print(missing[missing > 0] if (missing > 0).any() else 'Aucune valeur manquante')



Types de colonnes:
city_name                  object
country                    object
population_density          int64
avg_income                  int64
internet_penetration      float64
avg_rent                    int64
air_quality_index           int64
public_transport_score    float64
happiness_score           float64
green_space_ratio         float64
dtype: object

Valeurs manquantes par colonne:
Aucune valeur manquante


## 2. Preprocessing pour PCA

PCA fonctionne sur des variables numeriques comparables en echelle.
Dans cette etape, on:
- selectionne les colonnes numeriques
- verifie qu'il n'y a pas de valeurs manquantes sur ces colonnes
- standardise les features avec `StandardScaler`


In [5]:
# Selection des variables numeriques
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
X = df[numeric_cols].copy()

print(f'Colonnes numeriques ({len(numeric_cols)}): {numeric_cols}')
print(f'Shape de X: {X.shape}')

if X.isna().any().any():
    raise ValueError('Valeurs manquantes detectees dans les colonnes numeriques. Nettoyer avant PCA.')


Colonnes numeriques (8): ['population_density', 'avg_income', 'internet_penetration', 'avg_rent', 'air_quality_index', 'public_transport_score', 'happiness_score', 'green_space_ratio']
Shape de X: (300, 8)


In [6]:
# Standardisation des variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_cols)
X_scaled_df.describe().T[['mean', 'std']].head(10)


,mean,std
population_density,0.000000e+00,1.001671
avg_income,1.421085e-16,1.001671
internet_penetration,6.276461e-16,1.001671
avg_rent,3.552714e-17,1.001671
air_quality_index,-1.184238e-16,1.001671
public_transport_score,8.052818e-16,1.001671
happiness_score,-7.105427e-17,1.001671
green_space_ratio,1.184238e-16,1.001671
